# EVLM reproduction

Latency and accuracy of SmolVLM-256M on the EVLM paper (Farahani et al., EDGE 2026) datasets: the UZH power line clips and InsPLAD. The first part runs the fastest PyTorch setup and the llama.cpp (GGUF) setups. The last part is the log of earlier runs.

In [ ]:
import copy
import gc
import io
import json
import os
import re
import shutil
import subprocess
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, StaticCache
from transformers.modeling_outputs import BaseModelOutput

if not Path("evlm").exists():
    os.chdir("/content/edge-ai-smol-lvm-250-optimization")

EVLM = Path("evlm")
DATA = EVLM / "data"
RESULTS = EVLM / "results"
GGUF = EVLM / "gguf"
TRT = EVLM / "trt"
LLAMA = EVLM / "llama.cpp"
BIN = LLAMA / "build" / "bin"
RESULTS.mkdir(parents=True, exist_ok=True)

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
MAX_NEW = 64
PROMPTS = {
    "clip": "Is there a transmission tower in the green area?",
    "component": "What is the power-line component in the image?",
    "rust": "Is rust visible?",
}
PREPROCESS = {"slow": (False, "cpu"), "fast-cpu": (True, "cpu"), "fast-gpu": (True, "cuda")}
BEST = {"backend": "torch", "preprocess": "fast-gpu", "size": 1024, "decode": "graph", "stop": True, "trt": True, "filter": True}
BASELINE = {"backend": "torch", "preprocess": "slow", "size": 0, "decode": "bare", "stop": True, "trt": False, "filter": True}

In [ ]:
!pip install -q tensorrt onnx

## Data

In [ ]:
UZH = "https://download.ifi.uzh.ch/rpg/powerline_tracking_dataset/power_line_dataset.zip"
MENDELEY = "https://data.mendeley.com/public-files/datasets/5n3fjgvfyz/files/96707044-99bb-40b2-bf23-6fa1b41ab9b0/file_downloaded"
ENVS = {
    "forest": ("powerline_1_forest", 0.5),
    "city1": ("powerlines_16_city", 0.1),
    "mountain": ("powerline_7_mountain", 0.5),
    "city2": ("powerline_5_japan", 0.5),
    "sand": ("powerline_3_sand", 0.5),
    "warehouse": ("powerline_11_warehouse", 0.5),
    "wasteland": ("powerline_4_wasteland", 0.5),
    "industrial": ("powerline_2_industrial", 0.5),
}


class RemoteFile(io.RawIOBase):
    def __init__(self, src):
        self.src, self.pos = src, 0
        self.resolve()

    def resolve(self):
        req = urllib.request.Request(self.src, headers={"User-Agent": "curl/8", "Range": "bytes=0-0"})
        with urllib.request.urlopen(req) as r:
            self.url = r.url
            self.size = int(r.headers["Content-Range"].split("/")[1])

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = {0: off, 1: self.pos + off, 2: self.size + off}[whence]
        return self.pos

    def readinto(self, b):
        if self.pos >= self.size:
            return 0
        end = min(self.pos + len(b), self.size) - 1
        for attempt in range(2):
            req = urllib.request.Request(self.url, headers={"User-Agent": "curl/8", "Range": f"bytes={self.pos}-{end}"})
            try:
                with urllib.request.urlopen(req) as r:
                    data = r.read()
                break
            except urllib.error.HTTPError as e:
                if e.code != 403 or attempt:
                    raise
                self.resolve()
        b[: len(data)] = data
        self.pos += len(data)
        return len(data)


def remote_zip(url, buffer_mb=1):
    return zipfile.ZipFile(io.BufferedReader(RemoteFile(url), buffer_size=buffer_mb << 20))


def fetch_clips(frames=120):
    z = remote_zip(UZH)
    for env, (folder, where) in ENVS.items():
        pattern = re.compile(rf"datasets/{folder}/images/\w+/frame_(\d+)\.png$")
        names = {int(m.group(1)): m.group(0) for m in (pattern.match(i.filename) for i in z.infolist()) if m}
        order = sorted(names)
        start = max(0, int(len(order) * where) - frames // 2)
        dst = DATA / "powerline_clips" / env / "clip_0"
        dst.mkdir(parents=True, exist_ok=True)
        for i, n in enumerate(order[start:start + frames]):
            path = dst / f"{i:04d}.png"
            if not path.exists():
                path.write_bytes(z.read(names[n]))


def fetch_insplad():
    out = DATA / "insplad"
    if (out / "defect_supervised").exists():
        return
    out.mkdir(parents=True, exist_ok=True)
    inner = out / "supervised_fault_classification.zip"
    with remote_zip(MENDELEY, buffer_mb=8).open("supervised_fault_classification.zip") as src:
        with open(inner, "wb") as f:
            shutil.copyfileobj(src, f, 8 << 20)
    zipfile.ZipFile(inner).extractall(out)
    inner.unlink()


if not (DATA / "powerline_clips").exists():
    fetch_clips()
fetch_insplad()
print({env: len(list((DATA / "powerline_clips" / env / "clip_0").glob("*.png"))) for env in ENVS})

## Frame filter and InsPLAD samples

In [ ]:
def histogram(arr):
    q = arr >> 5
    h = np.bincount((q[..., 0].astype(np.int32) << 6 | q[..., 1] << 3 | q[..., 2]).ravel(), minlength=512)
    h = h.astype(np.float32)
    return h / (np.linalg.norm(h) + 1e-12)


def key_frames(paths, tau=0.95):
    keys, prev = [], None
    for p in paths:
        h = histogram(np.asarray(Image.open(p).convert("RGB")))
        if prev is None or np.corrcoef(h, prev)[0, 1] < tau:
            keys.append(p)
            prev = h
    return keys


def insplad_samples(limit=10):
    groups = {}
    for p in sorted((DATA / "insplad").glob("defect_supervised/*/val/*/*")):
        if p.suffix.lower() in (".jpg", ".jpeg", ".png"):
            groups.setdefault((p.parent.parent.parent.name, p.parent.name), []).append(p)
    return [(asset, label, p) for (asset, label), ps in sorted(groups.items()) for p in ps[:limit]]


def says_yes(answer):
    a = answer.lower()
    if a.startswith("no") or "no rust" in a or "not visible" in a or "no visible" in a:
        return False
    return a.startswith("yes") or "rust" in a

## Runners: PyTorch and llama.cpp

In [ ]:
def timed(fn):
    torch.cuda.synchronize()
    t = time.perf_counter()
    out = fn()
    torch.cuda.synchronize()
    return out, (time.perf_counter() - t) * 1e3


def until_eos(ids, eos):
    for i, t in enumerate(ids):
        if t in eos:
            return ids[:i]
    return ids


def set_attn(model, impl):
    if hasattr(model, "set_attn_implementation"):
        model.set_attn_implementation(impl)
    else:
        for c in (model.config.text_config, model.config.vision_config):
            c._attn_implementation = impl


class GraphDecoder:
    def __init__(self, model, max_new, max_len):
        self.model, self.max_new, self.max_len = model, max_new, max_len
        self.cache = StaticCache(config=model.config.text_config, max_cache_len=max_len)
        dev = model.device
        self.ids = torch.zeros(1, 1, dtype=torch.long, device=dev)
        self.pos = torch.zeros(1, dtype=torch.long, device=dev)
        self.idx = torch.zeros(1, dtype=torch.long, device=dev)
        self.out = torch.zeros(max_new, dtype=torch.long, device=dev)
        self.graph = None

    def step(self):
        h = self.model.model.text_model(
            input_ids=self.ids, cache_position=self.pos, past_key_values=self.cache, use_cache=True
        ).last_hidden_state
        nxt = self.model.lm_head(h[:, -1]).argmax(-1, keepdim=True)
        self.ids.copy_(nxt)
        self.out.index_copy_(0, self.idx, nxt.view(1))
        self.pos.add_(1)
        self.idx.add_(1)

    @torch.inference_mode()
    def capture(self):
        s = torch.cuda.Stream()
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):
                self.pos.fill_(self.max_len - 1)
                self.idx.zero_()
                self.step()
        torch.cuda.current_stream().wait_stream(s)
        self.pos.fill_(self.max_len - 1)
        self.idx.zero_()
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph):
            self.step()

    @torch.inference_mode()
    def prefill(self, inputs):
        self.cache.reset()
        n = inputs["input_ids"].shape[1]
        logits = self.model(
            **inputs, past_key_values=self.cache, use_cache=True,
            cache_position=torch.arange(n, device=self.model.device),
        ).logits
        first = logits[:, -1].argmax(-1, keepdim=True)
        self.ids.copy_(first)
        self.pos.fill_(n)
        self.idx.zero_()
        return first

    @torch.inference_mode()
    def decode(self, use_graph, eos=None):
        for i in range(self.max_new):
            if use_graph:
                self.graph.replay()
            else:
                self.step()
            if eos and self.ids.item() in eos:
                break
        return self.out.clone()


class SiglipTrt(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.vision = model.model.vision_model
        self.owner = [model]
        self.engines = {}

    def forward(self, pixel_values, patch_attention_mask=None, *args, **kwargs):
        if patch_attention_mask is not None and not bool(patch_attention_mask.all()):
            return self.vision(pixel_values, patch_attention_mask, *args, **kwargs)
        hidden = self.vision.embeddings(pixel_values, patch_attention_mask)
        n = hidden.shape[0]
        if n not in self.engines:
            self.engines[n] = engine_for(self.owner[0], self.vision, hidden)
        return BaseModelOutput(last_hidden_state=self.engines[n](hidden))


class SiglipEncoder(torch.nn.Module):
    def __init__(self, vision):
        super().__init__()
        self.encoder = vision.encoder
        self.norm = vision.post_layernorm

    def forward(self, x):
        return self.norm(self.encoder(inputs_embeds=x)[0])


def export_onnx(model, vision, x, path):
    set_attn(model, "eager")
    enc = copy.deepcopy(SiglipEncoder(vision)).half().eval()
    set_attn(model, "sdpa")
    with torch.no_grad():
        torch.onnx.export(enc, (x.half(),), str(path), dynamo=False,
                          input_names=["embeds"], output_names=["hidden"], opset_version=17)


def build_engine(onnx, engine):
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    nf = trt.NetworkDefinitionCreationFlag
    has_fp16 = hasattr(trt.BuilderFlag, "FP16")
    flag = 1 << int(nf.EXPLICIT_BATCH) if hasattr(nf, "EXPLICIT_BATCH") else 0
    if not has_fp16:
        flag |= 1 << int(nf.STRONGLY_TYPED)
    network = builder.create_network(flag)
    parser = trt.OnnxParser(network, logger)
    if not parser.parse(Path(onnx).read_bytes()):
        raise RuntimeError("; ".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
    config = builder.create_builder_config()
    if has_fp16:
        config.set_flag(trt.BuilderFlag.FP16)
    data = builder.build_serialized_network(network, config)
    if data is None:
        raise RuntimeError("TensorRT build failed")
    Path(engine).write_bytes(data)


class TrtEngine:
    def __init__(self, path):
        import tensorrt as trt

        engine = trt.Runtime(trt.Logger(trt.Logger.WARNING)).deserialize_cuda_engine(Path(path).read_bytes())
        self.ctx = engine.create_execution_context()
        names = [engine.get_tensor_name(i) for i in range(engine.num_io_tensors)]
        self.inp = next(n for n in names if engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT)
        self.out = next(n for n in names if engine.get_tensor_mode(n) == trt.TensorIOMode.OUTPUT)
        self.out_shape = tuple(engine.get_tensor_shape(self.out))
        dtypes = {trt.DataType.FLOAT: torch.float32, trt.DataType.HALF: torch.float16}
        self.in_dtype = dtypes[engine.get_tensor_dtype(self.inp)]
        self.out_dtype = dtypes[engine.get_tensor_dtype(self.out)]

    def __call__(self, x):
        x = x.to(self.in_dtype).contiguous()
        y = torch.empty(self.out_shape, dtype=self.out_dtype, device=x.device)
        self.ctx.set_tensor_address(self.inp, x.data_ptr())
        self.ctx.set_tensor_address(self.out, y.data_ptr())
        self.ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream)
        return y.half()


def engine_for(model, vision, hidden):
    TRT.mkdir(parents=True, exist_ok=True)
    n = hidden.shape[0]
    onnx, engine = TRT / f"siglip_{n}tiles.onnx", TRT / f"siglip_{n}tiles.engine"
    if not engine.exists():
        with torch.inference_mode(False):
            export_onnx(model, vision, hidden.clone(), onnx)
            build_engine(onnx, engine)
    return TrtEngine(engine)


class TorchVlm:
    def __init__(self, preprocess, size, decode, stop, trt, max_new=MAX_NEW):
        fast, self.image_device = PREPROCESS[preprocess]
        self.processor = AutoProcessor.from_pretrained(MODEL_ID, use_fast=fast)
        self.model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16).to("cuda").eval()
        tok = self.processor.tokenizer
        self.eos = {tok.eos_token_id, tok.convert_tokens_to_ids("<end_of_utterance>")}
        self.image_kwargs = {"size": {"longest_edge": size}} if size else {}
        self.decode, self.stop, self.max_new = decode, stop, max_new
        self.dec = None
        if trt:
            self.model.model.vision_model = SiglipTrt(self.model)

    def inputs(self, image, prompt):
        text = self.processor.apply_chat_template(
            [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}],
            add_generation_prompt=True,
        )
        device = {} if self.image_device == "cpu" else {"device": self.image_device}
        kw = {**self.image_kwargs, **device}
        try:
            out = self.processor(text=text, images=[image], return_tensors="pt", images_kwargs=kw)
        except (TypeError, ValueError):
            out = self.processor(text=text, images=[image], return_tensors="pt", **kw)
        out = out.to("cuda")
        out["pixel_values"] = out["pixel_values"].to(torch.float16)
        return out

    def decoder(self, n):
        if self.dec is None or n + self.max_new >= self.dec.max_len:
            max_len = -(-(n + self.max_new + 1) // 64) * 64
            self.dec = GraphDecoder(self.model, self.max_new, max_len)
            if self.decode == "graph":
                self.dec.capture()
        return self.dec

    def answer(self, inputs):
        n = inputs["input_ids"].shape[1]
        if self.decode == "bare":
            extra = {} if self.stop else {"min_new_tokens": self.max_new}

            def generate(k, **kw):
                with torch.inference_mode():
                    return self.model.generate(**inputs, max_new_tokens=k, do_sample=False, **kw)

            _, prefill_ms = timed(lambda: generate(1))
            out, vlm_ms = timed(lambda: generate(self.max_new, **extra))
            ids, decode_ms = out[0, n:].tolist(), max(vlm_ms - prefill_ms, 0.0)
        else:
            dec = self.decoder(n)
            first, prefill_ms = timed(lambda: dec.prefill(inputs))
            out, decode_ms = timed(lambda: dec.decode(self.decode == "graph", self.eos if self.stop else None))
            ids = torch.cat([first.view(1), out]).tolist()
            vlm_ms = prefill_ms + decode_ms
        return until_eos(ids, self.eos), prefill_ms, decode_ms, vlm_ms

    def ask(self, image, prompt):
        inputs, input_ms = timed(lambda: self.inputs(image, prompt))
        ids, prefill_ms, decode_ms, vlm_ms = self.answer(inputs)
        return {"answer": self.processor.tokenizer.decode(ids, skip_special_tokens=True).strip(),
                "tokens": len(ids), "input_ms": input_ms, "prefill_ms": prefill_ms,
                "decode_ms": decode_ms, "vlm_ms": vlm_ms}


GGUF_HF = "https://huggingface.co/ggml-org/SmolVLM-256M-Instruct-GGUF/resolve/main"
F16 = "SmolVLM-256M-Instruct-f16.gguf"
MMPROJ = "mmproj-SmolVLM-256M-Instruct-f16.gguf"
RE_ENCODE = re.compile(r"mtmd batch encoding done in ([\d.]+) ms")
RE_PREFILL = re.compile(r"prompt eval time =\s*([\d.]+) ms /\s*(\d+) tokens")
RE_DECODE = re.compile(r"(?<!prompt )eval time =\s*([\d.]+) ms /\s*(\d+) runs")


def ensure_llama():
    if (BIN / "llama-mtmd-cli").exists():
        return
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA)], check=True)
    subprocess.run(["cmake", "-S", str(LLAMA), "-B", str(LLAMA / "build"), "-DGGML_CUDA=ON",
                    "-DCMAKE_BUILD_TYPE=Release"], check=True)
    subprocess.run(["cmake", "--build", str(LLAMA / "build"), "--config", "Release", "-j", "2",
                    "--target", "llama-mtmd-cli", "llama-quantize"], check=True)


class GgufVlm:
    def __init__(self, quant, max_new=MAX_NEW, ngl=99):
        GGUF.mkdir(parents=True, exist_ok=True)
        for name in (F16, MMPROJ):
            if not (GGUF / name).exists():
                urllib.request.urlretrieve(f"{GGUF_HF}/{name}", GGUF / name)
        self.mmproj, self.model = GGUF / MMPROJ, GGUF / F16
        if quant != "f16":
            self.model = GGUF / f"SmolVLM-256M-Instruct-{quant.upper()}.gguf"
            if not self.model.exists():
                subprocess.run([str(BIN / "llama-quantize"), str(GGUF / F16), str(self.model), quant.upper()],
                               check=True, capture_output=True)
        self.max_new, self.ngl, self.tmp = max_new, ngl, GGUF / "current.png"

    def ask(self, image, prompt):
        image.save(self.tmp)
        cmd = [str(BIN / "llama-mtmd-cli"), "-m", str(self.model), "--mmproj", str(self.mmproj),
               "--image", str(self.tmp), "-p", prompt, "-n", str(self.max_new), "-ngl", str(self.ngl),
               "--temp", "0", "-v"]
        r = subprocess.run(cmd, capture_output=True, text=True, check=True)
        encode = sum(float(m) for m in RE_ENCODE.findall(r.stderr))
        prefill_ms = float(RE_PREFILL.findall(r.stderr)[-1][0])
        decode_ms, tokens = RE_DECODE.findall(r.stderr)[-1]
        return {"answer": r.stdout.strip(), "tokens": int(tokens), "input_ms": encode,
                "prefill_ms": prefill_ms, "decode_ms": float(decode_ms), "vlm_ms": prefill_ms + float(decode_ms)}


def make_vlm(setup):
    if setup["backend"] == "gguf":
        ensure_llama()
        return GgufVlm(setup["quant"])
    return TorchVlm(setup["preprocess"], setup["size"], setup["decode"], setup["stop"], setup["trt"])

## Runs

In [ ]:
def call_stats(calls):
    decode_ms = sum(c["decode_ms"] for c in calls)
    tokens = sum(c["tokens"] for c in calls)
    return {
        "vlm_ms_per_call": round(float(np.mean([c["input_ms"] + c["vlm_ms"] for c in calls])), 1),
        "prefill_ms": round(float(np.mean([c["prefill_ms"] for c in calls])), 1),
        "tokens_per_answer": round(float(np.mean([c["tokens"] for c in calls])), 2),
        "decode_ms_per_token": round(decode_ms / max(tokens, 1), 2),
    }


def run_clips(vlm, log, envs=None, use_filter=True):
    clips = [c for c in sorted((DATA / "powerline_clips").glob("*/clip_0")) if envs is None or c.parent.name in envs]
    vlm.ask(Image.open(sorted(clips[0].glob("*.png"))[0]).convert("RGB"), PROMPTS["clip"])
    calls, segments, kept = [], [], []
    for clip in clips:
        paths = sorted(clip.glob("*.png"))
        t = time.perf_counter()
        keys = key_frames(paths) if use_filter else paths
        filter_s = time.perf_counter() - t
        clip_calls = []
        for p in keys:
            c = vlm.ask(Image.open(p).convert("RGB"), PROMPTS["clip"])
            clip_calls.append(c)
            log({"task": "clips", "clip": f"{clip.parent.name}/{clip.name}", "frame": p.name, **c})
        calls += clip_calls
        segments.append(filter_s + sum(c["input_ms"] + c["vlm_ms"] for c in clip_calls) / 1e3)
        kept.append(len(keys))
    return {"clips": len(clips), "key_frames": round(float(np.mean(kept)), 2), **call_stats(calls),
            "segment_s": round(float(np.mean(segments)), 3)}


def run_insplad(vlm, log, limit=10):
    samples = insplad_samples(limit)
    vlm.ask(Image.open(samples[0][2]).convert("RGB"), PROMPTS["component"])
    comp, rust, calls = [], [], []
    for asset, label, path in samples:
        image = Image.open(path).convert("RGB")
        c = vlm.ask(image, PROMPTS["component"])
        r = vlm.ask(image, PROMPTS["rust"])
        words = [w for w in asset.split("-") if len(w) > 3 and w != "upper"]
        comp.append(any(w in c["answer"].lower() for w in words))
        if label in ("good", "rust"):
            rust.append(says_yes(r["answer"]) == (label == "rust"))
        calls += [c, r]
        log({"task": "insplad", "asset": asset, "label": label, "path": str(path), "component": c, "rust": r})
    return {"samples": len(samples), "component_acc": round(float(np.mean(comp)), 3),
            "rust_acc": round(float(np.mean(rust)), 3), **call_stats(calls)}


def evaluate(name, setup, tasks=("clips", "insplad"), envs=None):
    vlm = make_vlm(setup)
    summary = {"name": name, **setup}
    with open(RESULTS / f"{name}.jsonl", "w") as f:
        def log(record):
            f.write(json.dumps(record) + "\n")
        if "clips" in tasks:
            summary["clips"] = run_clips(vlm, log, envs, setup.get("filter", True))
        if "insplad" in tasks:
            summary["insplad"] = run_insplad(vlm, log)
        f.write(json.dumps({"summary": summary}) + "\n")
    del vlm
    gc.collect()
    torch.cuda.empty_cache()
    return summary


def clips_table(results):
    cols = ["key_frames", "vlm_ms_per_call", "prefill_ms", "tokens_per_answer", "decode_ms_per_token", "segment_s"]
    return pd.DataFrame({name: s["clips"] for name, s in results.items()}).T[cols]


def insplad_table(results):
    cols = ["component_acc", "rust_acc", "vlm_ms_per_call", "prefill_ms", "tokens_per_answer", "decode_ms_per_token"]
    return pd.DataFrame({name: s["insplad"] for name, s in results.items()}).T[cols]

### Best setup and the paper-like baseline, both tasks

In [ ]:
best = evaluate("best", BEST)
baseline = evaluate("paper-like baseline", BASELINE)
both = {"best": best, "paper-like baseline": baseline}
display(clips_table(both))
display(insplad_table(both))

### Paper comparison

The paper's Table V gives 6.448 s per segment and 3.45 key frames per segment, so about 1.87 s per call. Compare the time per call and the tokens per answer, not the segment time.

In [ ]:
paper = {"key_frames": 3.45, "vlm_ms_per_call": round(6448 / 3.45, 1), "segment_s": 6.448}
rows = {name: {"key_frames": s["clips"]["key_frames"], "vlm_ms_per_call": s["clips"]["vlm_ms_per_call"],
               "tokens_per_answer": s["clips"]["tokens_per_answer"], "segment_s": s["clips"]["segment_s"]}
        for name, s in both.items()}
rows["paper (EVLM, Jetson)"] = paper
display(pd.DataFrame(rows).T)

### Ablation (clips, forest and sand)

In [ ]:
ABLATION = {
    "best": {},
    "no TensorRT": {"trt": False},
    "default tiles": {"size": 0},
    "no stop at EOS": {"stop": False},
    "eager decode": {"decode": "eager"},
    "bare generate": {"decode": "bare"},
    "slow preprocess": {"preprocess": "slow"},
    "no frame filter": {"filter": False},
    "nothing (baseline)": {"preprocess": "slow", "size": 0, "decode": "bare", "trt": False},
}
ablation = {name: evaluate(f"ablation {name}", {**BEST, **change}, tasks=("clips",), envs=["forest", "sand"])
            for name, change in ABLATION.items()}
display(clips_table(ablation))

### llama.cpp (GGUF), every quant, both tasks

In [ ]:
gguf = {f"GGUF {q}": evaluate(f"gguf {q}", {"backend": "gguf", "quant": q, "filter": True})
        for q in ("f16", "q8_0", "q4_k_m")}
compare = {"PyTorch best": best, **gguf}
display(clips_table(compare))
display(insplad_table(compare))

## Log of previous runs

The cells below are the earlier notebook, kept with their outputs. They call the scripts that were in the repository before the move to `evlm/`, so they are not run again.

# SmolVLM-256M: EVLM Benchmark

This notebook runs the benchmark of the EVLM paper (Farahani et al., *EVLM: Intent-Driven Edge Vision Language Model for UAV-Based Power Line Inspection*, EDGE 2026) with the fastest inference setup of this repository.

The paper uses two public datasets (Section V-B):

- **Power line videos** (Xing et al., IROS 2023, ref. [35]): 720x480 frames at 30 fps from eight environments. The paper cuts them into 4 s clips of 120 frames. This dataset is used for the speed tests.
- **InsPLAD** (Vieira e Silva et al., 2023, ref. [43]): UAV images of power line parts with labels. We use the fault classification crops (good, rust and other defects). This dataset is used for the accuracy test.

`get_data.py` downloads only the parts that the test needs: one 4 s clip for each environment (about 0.5 GB) and the InsPLAD fault crops (about 0.9 GB). The full archives are 25 GB and 6.4 GB.

`evlm_eval.py` has two tasks:

1. `clips`: for each clip, a histogram filter keeps the key frames (512 bins, threshold 0.95, as in the paper). The VLM answers the operator intent for each key frame. The script prints the key frames, the filter time, the VLM time and the segment latency.
2. `insplad`: the VLM answers the two prompts of the paper's Table III for each image. The script scores the component name and the rust answer against the labels.

The defaults are the best setup from the earlier tests: fast GPU preprocessing, `--size 1024`, CUDA-graph decoding, stop at EOS, and a TensorRT engine for SigLIP. Each part has a flag, so the last cell can turn the parts off one at a time (ablation).

We use the base model. The paper fine-tunes it with LoRA, and its weights are not public, so the accuracy here is lower than in the paper.

In [3]:
!git -C /content/edge-ai-smol-lvm-250-optimization pull
!pwd

fatal: cannot change to '/content/edge-ai-smol-lvm-250-optimization': No such file or directory
/content


In [4]:
!git clone https://github.com/mahdikhashan/edge-ai-smol-lvm-250-optimization /content/edge-ai-smol-lvm-250-optimization

Cloning into '/content/edge-ai-smol-lvm-250-optimization'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 257 (delta 167), reused 156 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (257/257), 155.13 KiB | 7.76 MiB/s, done.
Resolving deltas: 100% (167/167), done.


In [7]:
REPO = "./edge-ai-smol-vlm-256-optimization"
DATA = f"{REPO}/data"
OUT = f"{REPO}/results"

# Colab: packages for the TensorRT engine. The Jetson has TensorRT in JetPack.
!pip install -q tensorrt onnx

# dataset setup: about 1.4 GB, only the files the test needs; files that exist already are skipped
!python {REPO}/get_data.py --out {DATA}

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
clips: forest (powerline_1_forest) clip_0: 120 frames from frame 11295
clips: city1 (powerlines_16_city) clip_0: 120 frames from frame 21107
clips: mountain (powerline_7_mountain) clip_0: 120 frames from frame 19277
clips: city2 (powerline_5_japan) clip_0: 120 frames from frame 16784
clips: sand (powerline_3_sand) clip_0: 120 frames from frame 14129
clips: warehouse (powerline_11_warehouse) clip_0: 120 frames from frame 27219
clips: wasteland (powerline_4_wasteland) clip_0: 120 frames from frame 15424
clips: industrial (powerline_2_industrial) clip_0: 120 frames from frame 12632
insplad: 

In [6]:
# speed test on the power line clips, best setup
!python {REPO}/evlm_eval.py --task clips --data {DATA} --out {OUT}

processor_config.json: 100% 68.0/68.0 [00:00<00:00, 391kB/s]
chat_template.json: 100% 429/429 [00:00<00:00, 1.92MB/s]
preprocessor_config.json: 100% 486/486 [00:00<00:00, 2.99MB/s]
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
config.json: 100% 7.35k/7.35k [00:00<00:00, 23.0MB/s]
tokenizer_config.json: 100% 28.2k/28.2k [00:00<00:00, 83.6MB/s]
vocab.json: 100% 801k/801k [00:00<00:00, 34.2MB/s]
merges.txt: 100% 466k/466k [00:00<00:00, 109MB/s]
added_tokens.json: 100% 4.74k/4.74k [00:00<00:00, 18.7MB/s]
special_tokens_map.json: 100% 1.07k/1.07k [00:00<00:00, 3.83MB/s]
tokenizer.json: 100% 3.55M/3.55M [00:00<00:00, 186MB/s]

model.safetensors: downloading bytes:  33% 169M/513M [00:01<00:01, 231MB/s, 13.4MB/s  ]  
model.safetensors: downloading bytes:  45% 229M/513M [00:01<00:01, 258MB/s, 18.1MB/s  ]
model.safetensors: downloading bytes:  5

In [7]:
# accuracy test on InsPLAD (10 images for each asset and label), best setup
!python {REPO}/evlm_eval.py --task insplad --data {DATA} --out {OUT} --limit 10

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
Loading weights: 100% 471/471 [00:01<00:00, 269.33it/s]
[transformers] LANCZOS resample requires torchvision >= 0.27 and processing on CPU; it is not supported on CUDA or for torchvision < 0.27. Falling back to BICUBIC which approximates LANCZOS. To match the original model exactly, upgrade torchvision and move the tensors to CPU before resizing, or use a PIL backend image processor instead.
[10/23/2026-20:43:36] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[10/23/2026-20:43:44] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So th

In [8]:
# ablation: turn off one part at a time (two environments, to keep the run short)
import json
import subprocess

ENVS = "forest,sand"
CONFIGS = {
    "best": [],
    "no TensorRT": ["--no-trt"],
    "default tiles": ["--size", "0"],
    "no stop at EOS": ["--no-stop"],
    "eager decode": ["--decode", "eager"],
    "bare generate": ["--decode", "bare"],
    "slow preprocess": ["--preprocess", "slow"],
    "no frame filter": ["--filter", "none"],
    "nothing (baseline)": ["--preprocess", "slow", "--size", "0", "--decode", "bare", "--no-trt"],
}

print(f"{'config':<20} {'key frames':>10} {'VLM ms/key':>10} {'segment s':>9}")
for name, flags in CONFIGS.items():
    r = subprocess.run(
        ["python", f"{REPO}/evlm_eval.py", "--task", "clips", "--data", DATA, "--out", OUT, "--envs", ENVS, *flags],
        capture_output=True, text=True,
    )
    line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
    if line is None:
        print(f"{name:<20} failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
        continue
    s = json.loads(line[len("SUMMARY "):])
    print(f"{name:<20} {s['key_frames']:10.2f} {s['vlm_ms_per_key']:10.1f} {s['segment_s']:9.2f}")

config               key frames VLM ms/key segment s
best                       1.00      240.1      1.97
no TensorRT                1.00      251.7      1.85
default tiles              1.00      408.0      2.05
no stop at EOS             1.00      510.1      2.12
eager decode               1.00      805.5      2.35
bare generate              1.00      847.4      2.44
slow preprocess            1.00      352.5      1.93
no frame filter          120.00      191.9     24.24
nothing (baseline)         1.00     1367.2      2.92


In [9]:
# ablation (new version: tokens and ms per token): turn off one part at a time (two environments, to keep the run short)
import json
import subprocess

ENVS = "forest,sand"
CONFIGS = {
    "best": [],
    "no TensorRT": ["--no-trt"],
    "default tiles": ["--size", "0"],
    "no stop at EOS": ["--no-stop"],
    "eager decode": ["--decode", "eager"],
    "bare generate": ["--decode", "bare"],
    "slow preprocess": ["--preprocess", "slow"],
    "no frame filter": ["--filter", "none"],
    "nothing (baseline)": ["--preprocess", "slow", "--size", "0", "--decode", "bare", "--no-trt"],
}

print(f"{'config':<20} {'key frames':>10} {'VLM ms/call':>11} {'tokens':>6} {'ms/token':>8} {'segment s':>9}")
for name, flags in CONFIGS.items():
    r = subprocess.run(
        ["python", f"{REPO}/evlm_eval.py", "--task", "clips", "--data", DATA, "--out", OUT, "--envs", ENVS, *flags],
        capture_output=True, text=True,
    )
    line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
    if line is None:
        print(f"{name:<20} failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
        continue
    s = json.loads(line[len("SUMMARY "):])
    print(f"{name:<20} {s['key_frames']:10.2f} {s['vlm_ms_per_call']:11.1f} {s['tokens_per_answer']:6.1f} "
          f"{s['decode_ms_per_token']:8.2f} {s['segment_s']:9.2f}")

config               key frames VLM ms/call tokens ms/token segment s
best                       1.00       221.4   19.5     6.90      0.54
no TensorRT                1.00       239.8   19.5     6.86      0.57
default tiles              1.00       389.5   19.5     8.26      0.71
no stop at EOS             1.00       506.6   19.5    21.26      0.84
eager decode               1.00       794.6   19.5    34.21      1.11
bare generate              1.00       829.5   19.5    36.04      1.16
slow preprocess            1.00       351.7   19.5     8.02      0.68
no frame filter          120.00       187.8   14.1     6.98     22.54
nothing (baseline)         1.00      1354.7   19.5    33.87      1.69


## Comparison with the paper (run on the Jetson Orin)

The paper ran on an NVIDIA Jetson (8-core CPU, 16 GB, 1024 CUDA cores, 32 Tensor cores). Run the next cell on that device. Times from another GPU cannot be compared with the paper.

The cell runs two setups on all 8 environments:

- `paper-like baseline`: the plain model path (slow preprocessing, default tiles, Hugging Face `generate`, no TensorRT). This is our closest copy of the paper's pipeline. The paper does not give its inference code.
- `best`: all the speed-ups.

Compare the time for each VLM call, not the time for each segment. The paper's filter keeps about 3.45 frames per segment, and our clips can keep a different number. The last column scales our segment to 3.45 key frames.

The paper's time for each call is an estimate: 6.448 s per segment / 3.45 calls = about 1.87 s.

Also compare the tokens per answer, because a longer answer takes more time. The paper uses a fine-tuned model, so its answers can be longer or shorter.

The segment time is the filter time plus the VLM calls. The PNG decode is not included, because the paper decodes video and we decode PNG files.

In [ ]:
# Orin comparison: run on the Jetson Orin, all 8 environments
import json
import subprocess

PAPER_SEGMENT_S, PAPER_KEY_FRAMES = 6.448, 3.45  # Table V (EVLM) and the FOT frames per segment
SETUPS = {
    "paper-like baseline": ["--preprocess", "slow", "--size", "0", "--decode", "bare", "--no-trt"],
    "best": [],
}

results = {}
for name, flags in SETUPS.items():
    r = subprocess.run(
        ["python", f"{REPO}/evlm_eval.py", "--task", "clips", "--data", DATA, "--out", OUT, *flags],
        capture_output=True, text=True,
    )
    line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
    if line is None:
        print(f"{name}: failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
        continue
    results[name] = json.loads(line[len("SUMMARY "):])

print(f"{'setup':<20} {'key frames':>10} {'VLM s/call':>10} {'prefill ms':>10} {'tokens':>6} {'ms/token':>8} "
      f"{'segment s':>9} {'segment s at 3.45 keys':>22}")
for name, s in results.items():
    at_paper = s["filter_s_per_segment"] + PAPER_KEY_FRAMES * s["vlm_ms_per_call"] / 1e3
    print(f"{name:<20} {s['key_frames']:10.2f} {s['vlm_ms_per_call'] / 1e3:10.2f} {s['prefill_ms']:10.1f} "
          f"{s['tokens_per_answer']:6.1f} {s['decode_ms_per_token']:8.2f} {s['segment_s']:9.2f} {at_paper:22.2f}")
print(f"{'paper (EVLM)':<20} {PAPER_KEY_FRAMES:10.2f} {PAPER_SEGMENT_S / PAPER_KEY_FRAMES:10.2f} {'-':>10} {'-':>6} "
      f"{'-':>8} {PAPER_SEGMENT_S:9.2f} {PAPER_SEGMENT_S:22.2f}")

if len(results) == 2:
    base, best = results["paper-like baseline"], results["best"]
    print(f"speed-up for each VLM call, best against baseline: {base['vlm_ms_per_call'] / best['vlm_ms_per_call']:.2f}x")

In [8]:
# llama.cpp (GGUF) on both EVLM tasks, every quant. The PyTorch best setup runs too, as the reference.
# Colab: builds llama.cpp with CUDA if it is missing (about 20-30 minutes; -j 2 avoids the out-of-memory kill).
import json
import os
import subprocess

print("repo:", REPO, "| data:", DATA)
# Colab: the packages the PyTorch path needs, and the datasets when they are missing (a fresh clone has no data/).
!pip install -q tensorrt onnx
if not (os.path.isdir(f"{DATA}/powerline_clips") and os.path.isdir(f"{DATA}/insplad")):
    subprocess.run(["python", f"{REPO}/get_data.py", "--out", DATA], check=True)

BIN = "./llama.cpp/build/bin"
subprocess.run(
    "test -f ./llama.cpp/build/bin/llama-mtmd-cli || ("
    "git clone --depth 1 https://github.com/ggml-org/llama.cpp && "
    "cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release && "
    "cmake --build llama.cpp/build --config Release -j 2 --target llama-mtmd-cli llama-quantize)",
    shell=True, check=True,
)

CONFIGS = [("PyTorch best", [])] + [
    (f"GGUF {q}", ["--backend", "gguf", "--quant", q, "--cli", f"{BIN}/llama-mtmd-cli",
                   "--quantize", f"{BIN}/llama-quantize", "--gguf-dir", f"{REPO}/gguf"])
    for q in ("f16", "q8_0", "q4_k_m")
]

summaries = {}
for task in ("clips", "insplad"):
    for name, flags in CONFIGS:
        r = subprocess.run(["python", f"{REPO}/evlm_eval.py", "--task", task, "--data", DATA, "--out", OUT, *flags],
                           capture_output=True, text=True)
        line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
        if line is None:
            print(f"{task} {name}: failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
            continue
        summaries[(task, name)] = json.loads(line[len("SUMMARY "):])

print("clips (8 environments, 4 s segments)")
print(f"{'setup':<14} {'key frames':>10} {'VLM ms/call':>11} {'prefill ms':>10} {'tokens':>6} {'ms/token':>8} {'segment s':>9}")
for name, _ in CONFIGS:
    s = summaries.get(("clips", name))
    if s:
        print(f"{name:<14} {s['key_frames']:10.2f} {s['vlm_ms_per_call']:11.1f} {s['prefill_ms']:10.1f} "
              f"{s['tokens_per_answer']:6.1f} {s['decode_ms_per_token']:8.2f} {s['segment_s']:9.2f}")

print("\ninsplad (110 images, component and rust prompts)")
print(f"{'setup':<14} {'component acc':>13} {'rust acc':>8} {'VLM ms/call':>11} {'prefill ms':>10} {'tokens':>6} {'ms/token':>8}")
for name, _ in CONFIGS:
    s = summaries.get(("insplad", name))
    if s:
        rust = f"{s['rust_acc']:8.3f}" if s["rust_acc"] is not None else f"{'-':>8}"
        print(f"{name:<14} {s['component_acc']:13.3f} {rust} {s['vlm_ms_per_call']:11.1f} {s['prefill_ms']:10.1f} "
              f"{s['tokens_per_answer']:6.1f} {s['decode_ms_per_token']:8.2f}")

repo: ./edge-ai-smol-vlm-256-optimization | data: ./edge-ai-smol-vlm-256-optimization/data
clips (8 environments, 4 s segments)
setup          key frames VLM ms/call prefill ms tokens ms/token segment s
PyTorch best         1.12       196.9       85.8   14.4     7.30      0.56
GGUF f16             1.12       331.2      159.5   12.9     4.73      0.70
GGUF q8_0            1.12       320.3      163.9   12.9     3.77      0.69
GGUF q4_k_m          1.12       352.0      193.9   12.0     3.84      0.74

insplad (110 images, component and rust prompts)
setup          component acc rust acc VLM ms/call prefill ms tokens ms/token
PyTorch best           0.000    0.767       288.6      209.7   10.1     7.24
GGUF f16               0.000    0.700       334.4      175.3   10.4     3.61
GGUF q8_0              0.000    0.722       332.4      181.1   10.4     2.99
GGUF q4_k_m            0.009    0.722       363.5      207.8   11.5     2.91


## Result: llama.cpp (GGUF) against PyTorch on the EVLM tasks

The test ran on a Colab GPU, with the paper's clips (8 environments, one 4 s segment each) and 110 InsPLAD images. Each row is one run of the whole task.

**Clips**

| Setup | Key frames | VLM ms/call | Prefill (ms) | Tokens | Decode (ms/token) | Segment (s) |
|---|---|---|---|---|---|---|
| PyTorch best | 1.12 | 196.9 | 85.8 | 14.4 | 7.30 | 0.56 |
| GGUF f16 | 1.12 | 331.2 | 159.5 | 12.9 | 4.73 | 0.70 |
| GGUF q8_0 | 1.12 | 320.3 | 163.9 | 12.9 | 3.77 | 0.69 |
| GGUF q4_k_m | 1.12 | 352.0 | 193.9 | 12.0 | 3.84 | 0.74 |

**InsPLAD**

| Setup | Component acc | Rust acc | VLM ms/call | Prefill (ms) | Tokens | Decode (ms/token) |
|---|---|---|---|---|---|---|
| PyTorch best | 0.000 | 0.767 | 288.6 | 209.7 | 10.1 | 7.24 |
| GGUF f16 | 0.000 | 0.700 | 334.4 | 175.3 | 10.4 | 3.61 |
| GGUF q8_0 | 0.000 | 0.722 | 332.4 | 181.1 | 10.4 | 2.99 |
| GGUF q4_k_m | 0.009 | 0.722 | 363.5 | 207.8 | 11.5 | 2.91 |

- **PyTorch best is the fastest per call.** It takes 197 ms on clips and 289 ms on InsPLAD. The GGUF runs take 320 to 364 ms. The gap comes from the prefill: PyTorch takes 86 to 210 ms, and GGUF takes 140 to 208 ms.
- **GGUF decodes about 2 times faster per token.** It takes 2.9 to 4.7 ms per token, against 7.3 and 7.2 ms for PyTorch. The PyTorch run uses CUDA graphs, so the decode speed-up comes from llama.cpp's own kernels.
- **The segment time follows the prefill.** PyTorch takes 0.56 s per segment, and GGUF takes 0.69 to 0.74 s.
- **The token counts differ.** PyTorch gives about 14 tokens per answer on clips, and GGUF gives 12 to 13. Use the ms per token column to compare decoding, since it does not depend on the answer length.
- **Quantization does not change the accuracy.** Rust accuracy is 0.70 to 0.72 for every GGUF quant, with 110 images. One image is about 1%, so these differences are within noise.
- **4-bit gives no speed gain.** Q4_K_M decodes at the same speed as Q8_0 (3.84 against 3.77 ms on clips, 2.91 against 2.99 on InsPLAD). Its prefill is slower (194 against 164 ms on clips, 208 against 181 ms on InsPLAD). This matches the earlier single-image result.
- **The component score is still broken.** It is near 0 for every setup, so it does not measure anything yet. It is a keyword check, and it needs to be replaced before the results are used.
- **These are Colab numbers, not Jetson numbers.** The comparison with the paper still needs the Orin run.

**Conclusion:** on the Colab GPU, the PyTorch best setup gives the fastest response per call. GGUF gives faster decoding and the same accuracy. If the answer is short, the prefill decides the total, so PyTorch wins. If the answer is long, GGUF may win. Test that on the Jetson, where memory bandwidth is lower and the decode may matter more.